# 0.0 Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sklearn.metrics as mt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# 0.1 - Load Datasets

In [2]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks\regressao
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [3]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [4]:
# Instanciar o modelo com parâmetros default (grau 1)
model_def = Pipeline([
    ('features', PolynomialFeatures(degree=1)),
    ('model', LinearRegression())
])

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [5]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.0461
MSE:  456.00
RMSE: 21.35
MAE:  17.00
MAPE: 865.32%


## Passo 3 — Performance na validação (default)

In [6]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   0.0399
MSE:  458.45
RMSE: 21.41
MAE:  17.04
MAPE: 868.25%


## Passo 4 — Ajuste de hiperparâmetros

In [7]:
print("--- Ensaio Isolado: Testando o Impacto dos Graus Polinomiais ---")
melhor_degree = 1
melhor_r2_val = -999

# Lista de graus para testar o ganho de complexidade
list_degrees = [1, 2, 3]

for dg in list_degrees:
    
    # Criamos o algoritmo composto de Regressão Polinomial Pura (Sem freios)
    model_polinomial_puro = Pipeline([
        ('features_polinomiais', PolynomialFeatures(degree=dg, include_bias=False)),
        ('regressor_linear_puro', LinearRegression())
    ])
    
    try:
        # Treinamento direto do Pipeline
        model_polinomial_puro.fit(X_train, y_train.values.ravel())
        
        # Predict
        yhat_train = model_polinomial_puro.predict(X_train)
        yhat_val = model_polinomial_puro.predict(X_val)
        
        # Métricas
        r2_train = mt.r2_score(y_train, yhat_train)
        r2_val = mt.r2_score(y_val, yhat_val)
        rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
        
        print(f"Grau Polinômio: {dg} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f}")
        
        if r2_val > melhor_r2_val:
            melhor_r2_val = r2_val
            melhor_degree = dg
            
    except Exception as e:
        print(f"Erro ao processar o Grau {dg}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO DE GRAUS:")
print(f"Melhor Grau Encontrado: {melhor_degree}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")

--- Ensaio Isolado: Testando o Impacto dos Graus Polinomiais ---
Grau Polinômio: 1 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41
Grau Polinômio: 2 | R² Treino: 0.0942 | R² Val: 0.0665 | RMSE Val: 21.11
Grau Polinômio: 3 | R² Treino: 0.1544 | R² Val: -0.0478 | RMSE Val: 22.37
-> VENCEDOR DO ENSAIO DE GRAUS:
Melhor Grau Encontrado: 2
Maior R² obtido na Validação: 0.0665



## Passo 5 — Performance com modelo tunado (treino e validação)

In [8]:
# Modelo com os melhores hiperparâmetros encontrados, treinado apenas em X_train
model_tunado = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=melhor_degree, include_bias=False)),
    ('regressor_linear_puro', LinearRegression())
])
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"Treino    → R²: {r2_train_tunado:.4f} | RMSE: {rmse_train_tunado:.2f} | MAE: {mae_train_tunado:.2f} | MAPE: {mape_train_tunado*100:.2f}%")
print(f"Validação → R²: {r2_val_tunado:.4f} | RMSE: {rmse_val_tunado:.2f} | MAE: {mae_val_tunado:.2f} | MAPE: {mape_val_tunado*100:.2f}%")

--- Performance com Modelo Tunado ---
Treino    → R²: 0.0942 | RMSE: 20.81 | MAE: 16.46 | MAPE: 835.05%
Validação → R²: 0.0665 | RMSE: 21.11 | MAE: 16.75 | MAPE: 854.79%


## Passo 6 — União treino + validação

In [9]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [10]:
# Retreino com os melhores hiperparâmetros sobre o conjunto final (treino + validação)
model_final = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=melhor_degree, include_bias=False)),
    ('regressor_linear_puro', LinearRegression())
])
model_final.fit(X_train_final, y_train_final.values.ravel())

,steps,"[('features_polinomiais', ...), ('regressor_linear_puro', ...)]"
,transform_input,None
,memory,None
,verbose,False
,degree,2
,interaction_only,False
,include_bias,False
,order,'C'
,fit_intercept,True
,copy_X,True
,tol,1e-06


## Passo 8 — Performance no teste

In [11]:
# Predição no conjunto de teste (única vez que X_test é tocado)
yhat_test = model_final.predict(X_test)

# Métricas no conjunto de TESTE
r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print("--- Performance no Teste (Final) ---")
print(f"R²:   {r2_test:.4f}")
print(f"MSE:  {mse_test:.2f}")
print(f"RMSE: {rmse_test:.2f}")
print(f"MAE:  {mae_test:.2f}")
print(f"MAPE: {mape_test * 100:.2f}%")

--- Performance no Teste (Final) ---
R²:   0.0909
MSE:  442.64
RMSE: 21.04
MAE:  16.74
MAPE: 827.70%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [12]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto       R²      RMSE       MAE    MAPE
   Treino (Default) 0.046058 21.354065 16.998249 865.32%
Validação (Default) 0.039925 21.411376 17.039754 868.25%
    Treino (Tunado) 0.094195 20.808321 16.458032 835.05%
 Validação (Tunado) 0.066477 21.113224 16.749939 854.79%
      Teste (Final) 0.090901 21.039044 16.736414 827.70%
